In [1]:
import sys
import os
import datetime as dt
from tqdm.notebook import tqdm
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as cplt
sys.path.insert(1, '/home/bzbauer/Documents/UArizona/HEP/Scripts/susypy')
import susypy as ss

%config Completer.use_jedi = False
%config IPCompleter.greedy=True

# Options

do_sigma = False

# Plotting palette definition

#color_palette = ["#648FFF", "#785EF0", "#DC267F",
#                "#FE6100", "#FFB000"]
                
#cmap_linear = cplt.LinearSegmentedColormap.from_list(
#    name="colorblind-ibm", colors=color_palette
#)

plt.rcParams["axes.prop_cycle"] = plt.cycler("color", plt.cm.Dark2.colors)

In [2]:
# Defining directories and file paths

input_name = "Bino-like_NLSP.txt"

susy_x = "/home/bzbauer/Programs/softsusy-4.1.22/softpoint.x"

input_dir = "/home/bzbauer/Documents/UArizona/HEP/Scripts/BinoNLSP/input"
output_dir = "/home/bzbauer/Documents/UArizona/HEP/Scripts/BinoNLSP/output"

plot_dir = "/home/bzbauer/Documents/UArizona/HEP/Scripts/BinoNLSP/Plots"

dtime = dt.datetime.now()
date_string = (dtime.date().isoformat() 
    + "_" + str(dtime.time().hour) 
    + "-" + str(dtime.time().minute))

# Read in the base SLHA file

slha = ss.SLHA(input_name,
                      susy_x,
                      in_dir=input_dir,
                      out_dir=output_dir)

# Scan through a range of mu

masses = np.linspace(450, 800, 20)
tanbs = np.linspace(10, 60, 50)

mass_scan = ss.scan_params(slha, ("EXTPAR", 23), [f"{m:.8E}" for m in masses], new_dir=date_string)
tanb_scan = ss.scan_params(slha, ("MINPAR", 3), [f"{tb:.8E}" for tb in tanbs], new_dir=date_string)

# Define the electroweakino PDGs and labels

electroweakinos = ["1000022", "1000023", "1000025", "1000035", "1000024", "1000037"]
labels_ew = [r"\tilde{\chi}_1^0", r"\tilde{\chi}_2^0", r"\tilde{\chi}_3^0", r"\tilde{\chi}_4^0", r"\tilde{\chi}_1^+", r"\tilde{\chi}_2^+"]

In [ ]:

# Plot electroweakino masses as a function of bino mass

fig, ax = plt.subplots()

fig, ax = ss.plot_scan(mass_scan, "EXTPAR", "23", "MASS", electroweakinos, abs_val=True, fig=fig, ax=ax, label_list=labels_ew)

ax.set_xlabel(r"$\mu$ [GeV]")
ax.set_ylabel("Mass [GeV]")
ax.set_title(r"Particle Masses vs. $\mu$")
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
ax.grid()
plt.tight_layout()
fig.savefig(plot_dir + "/" + date_string + "_electroweakinos_vs_higgsino_mass.png", dpi=300, bbox_inches="tight")
plt.close(fig)


In [ ]:
# Plot electroweakino masses as a function of bino mass

fig, ax = plt.subplots()

fig, ax = ss.plot_scan(tanb_scan, "MINPAR", "3", "MASS", electroweakinos, abs_val=True, fig=fig, ax=ax, label_list=labels_ew)

ax.set_xlabel(r"$\tan{\beta}$")
ax.set_ylabel("Mass [GeV]")
ax.set_title(r"Particle Masses vs. $\tan{\beta}$")
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
ax.grid()
plt.tight_layout()
fig.savefig(plot_dir + "/" + date_string + "_electroweakinos_vs_tanb.png", dpi=300, bbox_inches="tight")
plt.close(fig)

In [ ]:
# Compute the production cross sections for electroweakinos at LHC for bino NLSP
if do_sigma:
    m1s = []
    sigmas = [[],[],[],[],[],[]]
    unctys = [[],[],[],[],[],[]]

    for mass in tqdm(mass_scan):
        
        m = np.abs(ss.gather_data([mass], "EXTPAR", "23"))
        sigma_n3n4, uncty_n3n4 = mass.cross_section("1000025", "1000035")
        sigma_c2c2, uncty_c2c2 = mass.cross_section("1000037", "-1000037")
        sigma_n3c2, uncty_n3c2 = mass.cross_section("1000025", "1000037")
        sigma_n4c2, uncty_n4c2 = mass.cross_section("1000035", "1000037")
        sigma_n3c2n, uncty_n3c2n = mass.cross_section("1000025", "-1000037")
        sigma_n4c2n, uncty_n4c2n = mass.cross_section("1000035", "-1000037")

        # Need help understanding the total cross section that we are looking for

        m1s.append(m)
        sigmas[0].append(sigma_n3n4)
        sigmas[1].append(sigma_c2c2)
        sigmas[2].append(sigma_n3c2)
        sigmas[3].append(sigma_n4c2)
        sigmas[4].append(sigma_n3c2n)
        sigmas[5].append(sigma_n4c2n)

        unctys[0].append(uncty_n3n4)
        unctys[1].append(uncty_c2c2)
        unctys[2].append(uncty_n3c2)
        unctys[3].append(uncty_n4c2)
        unctys[4].append(uncty_n3c2n)
        unctys[5].append(uncty_n4c2n)


    labels_sig = [r"\tilde{\chi}_3^0 + \tilde{\chi}_4^0", r"\tilde{\chi}_2^+ + \tilde{\chi}_2^-", r"\tilde{\chi}_3^0 + \tilde{\chi}_2^+",
                  r"\tilde{\chi}_4^0 + \tilde{\chi}_2^+", r"\tilde{\chi}_3^0 + \tilde{\chi}_2^-", r"\tilde{\chi}_4^0 + \tilde{\chi}_2^-"]

    # Plot the total cross section for each  higgsino-like electroweakino 

    fig, ax = plt.subplots()
    #ax.set_prop_cycle('color', color_palette)
    for sigma, uncty, label in zip(sigmas, unctys, labels_sig):
        ax.errorbar(m1s, sigma, yerr=uncty, label=r"${}$".format(label))

    ax.set_xlabel(r"$\mu$ [GeV]")
    ax.set_ylabel(r"$\sigma$ [pb]")
    ax.set_title(r"Pair Production Cross $\sigma$ vs $\mu$")
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    ax.grid()
    plt.tight_layout()

    fig.savefig(plot_dir + "/" + date_string + "_electroweakino_sigma_vs_higgsino_mass.png", dpi=300, bbox_inches="tight")

In [ ]:
# Branching Ratios maybe include the decay width?

electroweakinos_adj = ["1000023", "1000025", "1000035", "1000037"]
labels_ew_adj = [r"\tilde{\chi}_2^0", r"\tilde{\chi}_3^0", r"\tilde{\chi}_4^0", r"\tilde{\chi}_2^+"]

for ew, name in zip(electroweakinos_adj, labels_ew_adj):

    fig, ax = plt.subplots()

    fig, ax = ss.plot_scan(tanb_scan, "MINPAR", "3", ew, "all", abs_val=True, fig=fig, ax=ax, col_y=0)
    ax.set_xlabel(r"$\tan{\beta}$ [GeV]")
    ax.set_ylabel("Branching Ratio")
    ax.set_title(r"${0}$ Branching Ratio vs $\tan{1}$".format(name, r"\beta"))
    ax.grid()
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    fig.savefig(plot_dir + "/" + date_string + "_" + name[1:].replace("^", "").replace("+", "")
                + "_BR_vs_tanb.png", dpi=300, bbox_inches="tight")
    plt.tight_layout()
    plt.close(fig)

In [3]:
min_check_600s = ss.scan_params(slha, ("EXTPAR", 23), [f"{600:.8E}"], new_dir=date_string)
min_check_600 = min_check_600s[0]

# sigma_n3n4, uncty_n3n4 = min_check_600.cross_section("1000025", "1000035")
# sigma_c2c2, uncty_c2c2 = min_check_600.cross_section("1000037", "-1000037")
# sigma_n3c2, uncty_n3c2 = min_check_600.cross_section("1000025", "1000037")
# sigma_n4c2, uncty_n4c2 = min_check_600.cross_section("1000035", "1000037")
# sigma_n3c2n, uncty_n3c2n = min_check_600.cross_section("1000025", "-1000037")
# sigma_n4c2n, uncty_n4c2n = min_check_600.cross_section("1000035", "-1000037")


In [4]:
int_luminosity = 300 # fb^-1


## Need to potentially sum the conjugate intermediate states  with the same final state. May need work
## Potentially add option to specify an intermediate state that needs to be encountered.

n3 = min_check_600.stable_brs("1000025")
n4 = min_check_600.stable_brs("1000035")
c2 = min_check_600.stable_brs("1000037")

In [8]:
print(max(min_check_600.gen_final_states(n3, n4).values()))

0.09619720266228825


{'W+ ~chi_1-': 0.308596122,
 'W- ~chi_1+': 0.308596122,
 'Z ~chi_10': 0.239796445,
 'Z W+ ~chi_1-': 0.026694324677416618,
 'Z W- ~chi_1+': 0.026694324677416618,
 'Z Z ~chi_10': 0.0037706491777615625,
 'Z h ~chi_10': 0.014031948596213952,
 'h ~chi_10': 0.0651844117,
 'h W+ ~chi_1-': 0.0024881459579877063,
 'h W- ~chi_1+': 0.0024881459579877063,
 'h Z ~chi_10': 0.0003514576833844462,
 'h h ~chi_10': 0.0013079010840044904}